# Разбиение данных и baseline

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Разделять обучение, выбор и финальную оценку и измерять постоянный baseline.

**Модель** — правило, которое по признакам выдаёт предсказание. **Обучение** строит это правило по примерам с известными ответами. Например, постоянная модель запоминает самый частый ответ train и затем выдаёт его любому новому объекту. У более сложной модели предсказание зависит от признаков, но разделение обучения и применения остаётся тем же.

## Обучение, выбор и проверка

Модель должна отвечать на новые объекты, а не только запоминать таблицу. Поэтому данные заранее разделены:

- **train (обучающая часть):** 12 объектов для построения модели;
- **validation (проверочная часть):** 6 объектов для выбора настроек;
- **test (итоговая проверка):** 6 объектов для финальной оценки выбранного решения.



## Читаем готовые части

Скачайте [gliders.csv](../../../data/gliders.csv) в рабочую папку ноутбука. Поле `split` заранее задаёт часть для каждой строки. Числа преобразуем по схеме; затем выбираем строки включением. Тест пока только отделяем, но не оцениваем на нём варианты.

In [ ]:
import csv
with open("gliders.csv", encoding="utf-8", newline="") as stream:
    rows = list(csv.DictReader(stream))
for row in rows:
    row["span_cm"] = float(row["span_cm"])
    row["mass_g"] = float(row["mass_g"])
train = [row for row in rows if row["split"] == "train"]
validation = [row for row in rows if row["split"] == "validation"]
test = [row for row in rows if row["split"] == "test"]
assert (len(train), len(validation), len(test)) == (12, 6, 6)
ids = [row["id"] for row in rows]
assert len(set(ids)) == len(ids)  # Каждый id относится к одной строке.


## Разобранный пример: постоянный ответ

Baseline — простое решение для сравнения. Посчитаем частоты классов только по train и выберем самый частый. При равенстве берём первую метку по алфавиту: обход `sorted(counts)` задаёт порядок, а строгое сравнение частот сохраняет первого победителя.

In [ ]:
counts = {}
for row in train:
    label = row["flight"]
    counts[label] = counts.get(label, 0) + 1
winner = None
for label in sorted(counts):
    if winner is None or counts[label] > counts[winner]:
        winner = label
assert winner == "дальний"


Оба класса встречаются в train по шесть раз. Словарь не подсматривает в validation. Accuracy — число совпадений, делённое на число объектов; длины списков должны совпадать и быть положительными.

In [ ]:
def accuracy(expected, predicted):
    if len(expected) == 0 or len(expected) != len(predicted):
        raise ValueError("Нужны непустые списки одинаковой длины")
    correct = 0
    for left, right in zip(expected, predicted):
        if left == right:
            correct += 1
    return correct / len(expected)

expected = [row["flight"] for row in validation]
predicted = [winner for row in validation]
assert accuracy(expected, predicted) == 0.5


## Что означает результат

0.5 здесь — три верных ответа из шести. При 99% объектов одного класса постоянный ответ мог бы дать 0.99, не распознавая второй класс вообще. Поэтому нужны baseline и понимание частот классов. Многократный выбор по тесту делает его частью разработки и лишает независимости.

## Разобранный пример: что выучила константа

В train шесть меток каждого класса, поэтому заранее выбранное правило алфавита отдаёт победу «дальний». На validation три таких ответа из шести правильны. Если изменить только validation на шесть «коротких», обученная константа останется прежней, а качество станет нулевым. Это показывает различие обучения и измерения.

Обратите внимание: мы уже заглянули в ответы validation, чтобы измерить качество, но не меняли по ним константу. Если начать выбирать самый частый класс по validation, это будет обучение на проверке. Test для финальной оценки по-прежнему не использован.

## Типичные ошибки

Не выбирайте константу по validation или test. Проверьте непустоту и одинаковые длины списков перед accuracy; правило равенства частот задайте заранее.

## Задачи

### 1. Базовая

По какой части выбирается постоянная метка baseline?

<details>
<summary>Подсказка</summary>

Это обучение простого решения.

</details>



### 2. Базовая

Что означает accuracy 0.5 на шести объектах?

<details>
<summary>Подсказка</summary>

Умножьте долю на число объектов.

</details>



### 3. Прикладная

Вычислите accuracy для [0,1,1,0] и [0,0,1,0].

<details>
<summary>Подсказка</summary>

Сравните четыре позиции.

</details>



### 4. Прикладная

Проверьте отсутствие общих id в train и validation.

<details>
<summary>Подсказка</summary>

Соберите множества ключей.

</details>



### 5. Олимпиадная

Почему нельзя выбрать наиболее частый класс по всем трём частям?

<details>
<summary>Подсказка</summary>

Правильные ответы проверки попадут в решение.

</details>



### 6. Олимпиадная

Почему одинаковые объекты в train и test создают слишком лёгкую проверку?

<details>
<summary>Подсказка</summary>

Модель может запомнить уже виденное.

</details>



## Итоги

Постоянный ответ выбран по частотам train, а его качество измерено на validation. Даже у такого простого решения обучение и проверка выполняют разные роли; test пока остаётся для финальной оценки.

## Самопроверка

Покажите, какие строки меняют константу, а какие только метрику. Что произойдёт на пустой проверочной части?

Для перехода дальше достаточно базовых и прикладных задач; олимпиадные можно отложить.


## Решения задач

### Задача 1

По train: даже выбор одной постоянной метки является обучением и не использует проверочные ответы.

### Задача 2

0.5·6=3 верных ответа. Одна ошибка на таком наборе меняет accuracy на 1/6.

### Задача 3

In [ ]:
assert accuracy([0, 1, 1, 0], [0, 0, 1, 0]) == 0.75


### Задача 4

In [ ]:
train_ids = {row["id"] for row in train}
validation_ids = {row["id"] for row in validation}
assert train_ids & validation_ids == set()


### Задача 5

Это утечка: baseline уже использует метки объектов, на которых его оценивают.

### Задача 6

Проверка перестаёт моделировать появление новых объектов. Нужно разделять данные по реальным объектам или группам.